In [ ]:
import pandas as pd
import sys
import os
import numpy as np
from tqdm.auto import tqdm
tqdm.pandas()

sys.path.append(os.path.abspath("../.."))
from optimizer import variables

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

c:\Users\Daniel\venv-optimizer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def create_bess_designs(
    duration_low_h,
    duration_high_h,
    duration_step_h,
    power_low_mw,
    power_high_mw,
    power_step_mw,
    interval_minutes,
    usable_fraction,
    output_csv_path="2_Processed_data/bess_designs.csv",
    round_dp=4,
):

    def _inclusive_range(low, high, step, ndigits):
        values = []
        current = float(low)
        eps = 10 ** (-(ndigits + 2))
        while current <= float(high) + eps:
            values.append(round(current, ndigits))
            current += float(step)
        return np.array(values, dtype=float)

    durations = _inclusive_range(duration_low_h, duration_high_h, duration_step_h, round_dp)
    powers = _inclusive_range(power_low_mw, power_high_mw, power_step_mw, round_dp)

    rows = []
    interval_hours = interval_minutes / 60.0

    for duration_h in durations:
        for power_mw in powers:
            bess_mwh = power_mw * duration_h * usable_fraction
            bess_mwh_per_interval = power_mw * interval_hours

            rows.append({
                "BESS_Duration_h": round(float(duration_h), round_dp),
                "BESS_Power_MW": round(float(power_mw), round_dp),
                "BESS_MWh": round(float(bess_mwh), round_dp),
                "BESS_MWh_per_interval": round(float(bess_mwh_per_interval), round_dp),
            })

    bess_designs_df = pd.DataFrame(rows).drop_duplicates().sort_values(
        ["BESS_Duration_h", "BESS_Power_MW"]
    ).reset_index(drop=True)

    bess_designs_df.to_csv(output_csv_path, index=False)

    return bess_designs_df


# bess_designs_df = create_bess_designs(
#     duration_low_h=2,
#     duration_high_h=8.0,
#     duration_step_h=0.5,
#     power_low_mw=4.96,
#     power_high_mw=4.96,
#     power_step_mw=4.96,
#     interval_minutes=variables.operation_granularity_in_minutes,
#     usable_fraction=variables.bess_hours_usable_fraction,
#     output_csv_path="2_Processed_data/bess_designs.csv",
# )

# bess_designs_df

In [8]:
def create_scaled_generation_sets(
    power_low_mw,        
    power_high_mw,         
    power_step_mw,        
    input_csv_path,        
    output_csv_path,      
    round_dp=4,           
):

    # Create an inclusive float range helper (includes high bound when aligned).
    def _inclusive_range(low, high, step, ndigits):
        values = []                        # Collect generated values.
        current = float(low)               # Start at lower bound.
        eps = 10 ** (-(ndigits + 2))       # Tiny tolerance for float comparisons.
        while current <= float(high) + eps:
            values.append(round(current, ndigits))  # Store rounded value.
            current += float(step)         # Move to next value by step.
        return np.array(values, dtype=float)  # Return as numpy array for iteration.

    generation_df = pd.read_csv(input_csv_path)
    base_generation = pd.to_numeric(generation_df["Generation"], errors="coerce")
    base_generation = np.clip(base_generation.to_numpy(dtype=float), a_min=0.0, a_max=None)
    max_generation = float(np.max(base_generation)) if len(base_generation) else 0.0
    normalized_profile = np.clip(base_generation / max_generation, a_min=0.0, a_max=1.0)
    power_values = _inclusive_range(power_low_mw, power_high_mw, power_step_mw, round_dp)

    scaled = {}
    for power_mw in power_values:
        col_name = f"{power_mw:g}"             
        scaled[col_name] = np.round(normalized_profile * float(power_mw), round_dp)  

    scaled_df = pd.DataFrame(scaled)
    scaled_df.to_csv(output_csv_path, index=False)
    return scaled_df


# scaled_generation_df = create_scaled_generation_sets(
#     power_low_mw=0.0,
#     power_high_mw=7.0,
#     power_step_mw=1.0,
#     input_csv_path="2_Processed_data/generation.csv",
#     output_csv_path="2_Processed_data/resampled_generation_scaled.csv",
# )

# scaled_generation_df[:100]